# 01 — Kiểm định và tùy chọn tải dữ liệu FPT

Tương ứng src/fetch_data.py. Mặc định dùng snapshot cố định để tái lập báo cáo. Phần tải API chỉ chạy khi bạn chủ động bật.

**Cách chạy:** chọn kernel Python rồi chạy từng ô từ trên xuống (Shift+Enter), hoặc chọn **Run All**.

**Cần có trước:** data/raw/fpt/fpt_historical_price.csv đã nằm trong repo.

Các ô gọi hàm thật trong `src/`, không gọi `main()` nhận tham số của kernel và không chép lại thuật toán. Mỗi lần chạy có thể cập nhật tệp kết quả trong `data/processed/`, `reports/`, `figures/`, `models/`. Dữ liệu nghiên cứu dừng tại 30/09/2025.

Xem [hướng dẫn](README.md) nếu chạy trên Google Colab hoặc gặp lỗi kernel.

## 1. Tìm thư mục dự án

Chạy ô này trước các ô còn lại. Trên Colab, nếu chưa có dự án thì ô sẽ clone repo công khai; trên máy cá nhân, mở notebook từ bản clone đầy đủ.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Notebook không có __file__: tìm dự án từ thư mục đang mở và các thư mục cha.
IS_COLAB = "google.colab" in sys.modules
def is_project(folder):
    return (folder / "src" / "train.py").is_file() and (folder / "data" / "raw" / "fpt" / "fpt_historical_price.csv").is_file()

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((folder for folder in candidates if is_project(folder)), None)
if ROOT is None and IS_COLAB:
    # Colab chỉ có notebook; tải cả repo để có mã nguồn và dữ liệu đi kèm.
    clone_dir = Path("/content/fpt-stock-trends-prediction")
    if not clone_dir.exists():
        subprocess.check_call(["git", "clone", "https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git", str(clone_dir)])
    if is_project(clone_dir):
        ROOT = clone_dir
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook bên trong bản clone đầy đủ của dự án; xem notebooks/README.md.")
ROOT = ROOT.resolve()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Thư mục dự án:", ROOT)
print("Python của kernel:", sys.executable)

## 2. Kiểm tra môi trường

Local: cài `python -m pip install -r requirements-notebooks.txt` trước khi mở notebook. Colab: ô này cài phụ thuộc khi chạy. Nếu trình cài yêu cầu khởi động lại runtime, khởi động lại rồi chạy lại từ ô đầu.

In [ ]:
# Local: cài requirements-notebooks.txt trước khi mở Jupyter, để cờ False.
# Colab: mặc định cài thư viện xử lý từ requirements.txt, dùng CPU là đủ.
INSTALL_DEPENDENCIES = IS_COLAB
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])
else:
    print("Dùng các thư viện đã cài trong kernel hiện tại.")

## 3. Đường dẫn và cách xem kết quả

Giá có đơn vị nghìn VND/cổ phiếu; khối lượng là số cổ phiếu. Đường dẫn là biến Python để có thể xem ngay nơi đầu vào/đầu ra được lưu.

In [ ]:
import json
import io
from contextlib import redirect_stdout
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Mọi notebook dùng cùng cutoff và cùng nơi lưu kết quả như src/*.py.
CUTOFF = "2025-09-30"
PRICE_PATH = ROOT / "data/raw/fpt/fpt_historical_price.csv"
RAW_SUPPLEMENTAL_DIR = ROOT / "data/raw/supplemental"
PROCESSED_DIR = ROOT / "data/processed"
SUPPLEMENTAL_PATH = PROCESSED_DIR / "fpt_multisource_features.csv"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"
for folder in (PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def run_with_log(function, *args, **kwargs):
    # Hàm gốc in JSON dài. Lưu nhật ký đầy đủ, bên dưới xem bảng dễ đọc hơn.
    log = io.StringIO()
    try:
        with redirect_stdout(log):
            result = function(*args, **kwargs)
    finally:
        log_path = ROOT / "tmp/notebook_logs" / f"{function.__name__}.txt"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(log.getvalue(), encoding="utf-8")
    print("Nhật ký:", log_path.relative_to(ROOT))
    return result

def show_figure(name, width=1000):
    path = FIGURES_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}; chạy ô tạo biểu đồ hoặc notebook bước trước.")
    display(Image(filename=str(path), width=width))

## 4. Đọc và kiểm định snapshot

time là ngày; open/high/low/close là nghìn VND/cổ phiếu; volume là số cổ phiếu. Các bất nhất OHLC được ghi nhận, không tự ý sửa giá.

In [ ]:
from fetch_data import verify_snapshot
verification = verify_snapshot(PRICE_PATH, expected_last_date=CUTOFF)
display(pd.Series(verification, name="Giá trị kiểm định").to_frame())
prices = pd.read_csv(PRICE_PATH, parse_dates=["time"])
display(prices.head(3))
display(prices.tail(3))

## 5. Chuẩn hóa tên cột và kiểm tra kiểu

normalize_ohlcv chuyển ngày/giá/khối lượng về một schema thống nhất; lỗi trùng ngày, giá không hợp lệ sẽ được báo.

In [ ]:
from fetch_data import normalize_ohlcv, audit_ohlcv
normalized = normalize_ohlcv(pd.read_csv(PRICE_PATH))
display(normalized.dtypes.to_frame("Kiểu dữ liệu"))
display(pd.Series(audit_ohlcv(normalized)).to_frame("Kiểm toán"))

## 6. Tải lại từ VCI — tùy chọn

Giữ DOWNLOAD_FRESH=False để Run All offline. Nếu bật True, cần mạng/quyền vnstock. File mới được lưu riêng trong tmp/, không ghi đè snapshot nghiên cứu. API có thể chỉ trả một đoạn lịch sử hoặc giá đã hiệu chỉnh; kiểm tra độ phủ trước khi thay dữ liệu chính thức.

In [ ]:
from fetch_data import download_via_vnstock
DOWNLOAD_FRESH = False
if DOWNLOAD_FRESH:
    fresh = download_via_vnstock(start="2007-01-01", end=CUTOFF)
    download_path = ROOT / "tmp/notebook_download.csv"
    download_path.parent.mkdir(parents=True, exist_ok=True)
    fresh.to_csv(download_path, index=False)
    display(pd.Series(audit_ohlcv(fresh)).to_frame("Dữ liệu API thực nhận"))
    print("File tải thử:", download_path)
else:
    print("Không tải lại. Snapshot nghiên cứu vẫn được giữ nguyên.")

**Tiếp theo:** mở 02_fetch_supplemental.ipynb. Xem mã xử lý đầy đủ tại ../src/fetch_data.py.